<a href="https://colab.research.google.com/github/ridahafeez786/AI_Bootcamp_Lab-work/blob/main/Transformer_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Section 1: The Basic Idea — Simulated Memory Comparison

**Concept:** In real attention, every "head" (worker) normally keeps its own Key and Value. Multi-Query Attention (MQA) instead makes all heads share ONE Key and Value. These first 3 cells don't do any real attention math — they just simulate how much memory each approach would use, using random numbers as stand-ins.


In [1]:
# ============================================================
# Standard Multi-Head Attention (SIMULATION ONLY - not real attention)
# Every head has its own Key and Value cache.
# ============================================================

import numpy as np  # library for working with number arrays

num_heads = 8     # we're pretending we have 8 "workers" (heads)
seq_len = 1024     # pretend sentence is 1024 words long
head_dim = 64      # each worker's notes are 64 numbers long

print("="*60)
print("STANDARD MULTI-HEAD ATTENTION")
print("="*60)

total_memory = 0   # running total of memory used, starts at 0

for head in range(num_heads):  # repeat this block once per head (8 times total)

    # each head gets its OWN random Key and Value table (this is the "everyone
    # writes their own notebook" idea - nothing is shared here)
    K = np.random.randn(seq_len, head_dim)
    V = np.random.randn(seq_len, head_dim)

    # ask numpy how many actual bytes these two arrays take up in memory
    memory = K.nbytes + V.nbytes
    total_memory += memory   # add this head's memory to the running total

    print(f"Head {head+1}")
    print(f"Key Shape   : {K.shape}")
    print(f"Value Shape : {V.shape}")
    print(f"Memory Used : {memory/1024:.2f} KB")
    print("-"*40)

# convert total bytes into MB and print the grand total
# (this will be roughly 8x one head's memory, since 8 separate copies were made)
print(f"\nTotal KV Cache Memory = {total_memory/1024/1024:.2f} MB")


STANDARD MULTI-HEAD ATTENTION
Head 1
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 2
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 3
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 4
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 5
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 6
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 7
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 8
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
---------------------------

**What to notice in the output:** the total memory is exactly 8x one head's memory — because the loop created 8 completely separate K/V tables, one per head. Nothing is shared.


In [2]:
# ============================================================
# Multi Query Attention (MQA) (SIMULATION ONLY - not real attention)
# Every head has its own Query
# But ALL heads share ONE Key and ONE Value.
# ============================================================

import numpy as np

num_heads = 8
seq_len = 1024
head_dim = 64

print("="*60)
print("MULTI QUERY ATTENTION (MQA)")
print("="*60)

# Shared Key and Value — created ONCE, outside any loop, before any head runs.
# This is the whole trick: only 1 notebook is ever written, not 8.
shared_K = np.random.randn(seq_len, head_dim)
shared_V = np.random.randn(seq_len, head_dim)

# memory for just this ONE shared K/V pair
shared_memory = shared_K.nbytes + shared_V.nbytes

total_memory = shared_memory   # the total memory is JUST this one shared pair

print("Shared Key Shape   :", shared_K.shape)
print("Shared Value Shape :", shared_V.shape)
print()

for head in range(num_heads):   # still loop 8 times, but only for Query

    # each head STILL gets its own personal Query (this part isn't shared)
    Q = np.random.randn(seq_len, head_dim)

    print(f"Head {head+1}")
    print(f"Query Shape : {Q.shape}")
    print("Uses Shared Key")     # just a printed note - no new K is made here
    print("Uses Shared Value")   # just a printed note - no new V is made here
    print("-"*35)

# total memory here is just the ONE shared K/V pair (much smaller than Cell 1's total)
print(f"\nTotal KV Cache Memory = {total_memory/1024/1024:.2f} MB")


MULTI QUERY ATTENTION (MQA)
Shared Key Shape   : (1024, 64)
Shared Value Shape : (1024, 64)

Head 1
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 2
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 3
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 4
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 5
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 6
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 7
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 8
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------

Total KV Cache Memory = 1.00 MB


**What to notice:** total memory here is 8x smaller than the Standard version — because only ONE K/V pair was ever created, and all 8 heads just reused it. Only the Query stayed separate per head.


In [3]:
# ============================================================
# Final comparison: put the two memory numbers side by side
# (these numbers are just typed in directly from what we saw above -
#  no new calculation happens here, just a summary printout)
# ============================================================

standard_memory = 8      # MB  (from Cell 1's result)
mqa_memory = 1           # MB  (from Cell 2's result)

print("="*60)
print("MEMORY COMPARISON")
print("="*60)

print(f"Standard Attention : {standard_memory} MB")
print(f"MQA                : {mqa_memory} MB")
print()

# calculate what percentage smaller MQA's memory is compared to Standard
saving = (1 - mqa_memory / standard_memory) * 100

print(f"Memory Saved : {saving:.1f}%")


MEMORY COMPARISON
Standard Attention : 8 MB
MQA                : 1 MB

Memory Saved : 87.5%


---
## Section 2: Real Attention Math (PyTorch) — Standard vs MQA

**Concept:** the cells above were just memory simulations using random numbers. These next cells build the ACTUAL attention calculation using PyTorch — Query, Key, Value projections, real score computation, softmax, and blending — the full real mechanism, timed and measured for real.


In [4]:
# ============================================================
# Block 1: Import Libraries
# ============================================================

import torch                      # the main deep learning library
import torch.nn as nn              # building blocks for neural network layers
import torch.nn.functional as F    # extra functions like softmax
import time                        # for measuring how long code takes to run

torch.manual_seed(42)   # makes "random" numbers repeatable every time we run this

# check if a GPU is available; use it if so, otherwise fall back to normal CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)


Using Device: cuda


In [5]:
# ============================================================
# Block 2: Dummy Input
# ============================================================

batch_size = 2      # pretend we have 2 sentences at once
seq_len = 128        # each sentence has 128 words
embed_dim = 512      # each word is represented by 512 numbers
num_heads = 8        # we'll split work across 8 heads (workers)

# create a block of RANDOM numbers shaped like a real batch of sentences
# this stands in for real word embeddings, just so we can test our code
x = torch.randn(batch_size, seq_len, embed_dim).to(device)

print("Input Shape :", x.shape)


Input Shape : torch.Size([2, 128, 512])


**Concept for the next cell:** this builds the actual Multi-Head Attention "machine" as a reusable class. Setup happens once (`__init__`); the real math happens every time you run data through it (`forward`).


In [6]:
# ============================================================
# Block 3: Standard Multi-Head Attention
# ============================================================

class MultiHeadAttention(nn.Module):   # defining our own reusable attention "machine"

    def __init__(self, embed_dim=512, num_heads=8):
        # this part runs ONCE, when we first build the machine

        super().__init__()   # required PyTorch setup step

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads   # 512 / 8 = 64 numbers per head

        # three small "calculators" (learnable linear layers) that turn a word's
        # 512 numbers into a Query, Key, and Value - each still 512 numbers wide
        # (containing all 8 heads' worth, to be split apart later)
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # one more small calculator that runs at the very end, after attention
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # this part runs EVERY TIME we feed data through the machine

        B, T, C = x.shape   # B=batch size, T=sequence length, C=embed_dim - just labels

        # run the input through our three calculators to get Query, Key, Value
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # split each one's 512 numbers into 8 separate head-sized chunks of 64,
        # and rearrange dimensions so PyTorch can process all 8 heads at once
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        # compare every word's Query against every word's Key -> a score per pair
        # dividing by sqrt(head_dim) just keeps the numbers from growing too large
        scores = (Q @ K.transpose(-2,-1)) / (self.head_dim**0.5)

        # turn each word's row of scores into percentages that add up to 100%
        attn = F.softmax(scores, dim=-1)

        # blend every word's Value together, weighted by those percentages
        out = attn @ V

        # glue all 8 heads' separate results back into one single 512-number vector
        out = out.transpose(1,2).contiguous()
        out = out.view(B,T,C)

        # run through the final calculator before returning the answer
        return self.out(out)


In [7]:
# ============================================================
# Block 4: Run Standard Attention
# ============================================================

# actually build the machine using our chosen settings, and place it on GPU/CPU
mha = MultiHeadAttention(embed_dim, num_heads).to(device)

start = time.time()          # note the time before running

output = mha(x)               # run our fake sentence through the machine

if device == "cuda":
    torch.cuda.synchronize()  # wait for the GPU to actually finish before timing

end = time.time()             # note the time after running

print("="*60)
print("STANDARD MULTI-HEAD ATTENTION")
print("="*60)

print("Output Shape :", output.shape)
print("Runtime      :", round((end-start)*1000,2),"ms")

# add up every single number inside every calculator in the machine
# (this tells us how many "adjustable" numbers the whole machine has)
params = sum(p.numel() for p in mha.parameters())

print("Parameters   :", params)


STANDARD MULTI-HEAD ATTENTION
Output Shape : torch.Size([2, 128, 512])
Runtime      : 477.3 ms
Parameters   : 1050624


**Concept for the next cell:** this is the SAME structure as above, except the Key and Value calculators now only produce ONE head's worth of numbers (64, not 512) — and that single copy is then stretched (not duplicated) to be used by all 8 heads.


In [8]:
# ============================================================
# Block 5: Multi Query Attention
# ============================================================

class MultiQueryAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.Wq = nn.Linear(embed_dim, embed_dim)   # Query stays full-size (one per head)

        # Shared Key and Value - notice these only output head_dim (64) numbers,
        # NOT embed_dim (512) - because only ONE copy will exist, not 8
        self.Wk = nn.Linear(embed_dim, self.head_dim)
        self.Wv = nn.Linear(embed_dim, self.head_dim)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B,T,C = x.shape

        Q = self.Wq(x)

        K = self.Wk(x)
        V = self.Wv(x)

        # split Query into 8 heads, same as before
        Q = Q.view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        # Share K and V across all heads:
        # unsqueeze(1) adds a "heads" slot of size 1, then expand() makes it
        # LOOK like there are 8 copies - but no new memory is actually used,
        # it's the same underlying numbers viewed 8 times
        K = K.unsqueeze(1).expand(-1,self.num_heads,-1,-1)
        V = V.unsqueeze(1).expand(-1,self.num_heads,-1,-1)

        # everything from here is identical math to Standard Attention
        scores = (Q @ K.transpose(-2,-1))/(self.head_dim**0.5)

        attn = F.softmax(scores,dim=-1)

        out = attn @ V

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)


In [9]:
# ============================================================
# Block 6: Run MQA
# ============================================================

mqa = MultiQueryAttention(embed_dim, num_heads).to(device)

start = time.time()

output2 = mqa(x)

if device == "cuda":
    torch.cuda.synchronize()

end = time.time()

print("="*60)
print("MULTI QUERY ATTENTION")
print("="*60)

print("Output Shape :", output2.shape)
print("Runtime      :", round((end-start)*1000,2),"ms")

params = sum(p.numel() for p in mqa.parameters())

print("Parameters   :", params)


MULTI QUERY ATTENTION
Output Shape : torch.Size([2, 128, 512])
Runtime      : 76.08 ms
Parameters   : 590976


In [10]:
# ============================================================
# Block 7: Parameter Comparison
# (just comparing numbers already found in the two cells above)
# ============================================================

mha_params = sum(p.numel() for p in mha.parameters())
mqa_params = sum(p.numel() for p in mqa.parameters())

print("="*70)
print("PARAMETER COMPARISON")
print("="*70)

print(f"Standard MHA : {mha_params:,}")
print(f"MQA          : {mqa_params:,}")

saved = mha_params - mqa_params    # how many fewer numbers MQA needed

print(f"\nParameters Saved : {saved:,}")
print(f"Reduction        : {saved/mha_params*100:.2f}%")


PARAMETER COMPARISON
Standard MHA : 1,050,624
MQA          : 590,976

Parameters Saved : 459,648
Reduction        : 43.75%


In [11]:
# ============================================================
# Block 8: KV Cache Comparison
# (a manual formula estimating cache size - not measuring real memory directly)
# ============================================================

head_dim = embed_dim // num_heads

# MHA: needs to remember embed_dim (512) worth of numbers per word (all 8 heads)
mha_kv = batch_size * seq_len * embed_dim * 2   # the "2" = one K + one V

# MQA: only needs head_dim (64) worth of numbers per word (just the 1 shared copy)
mqa_kv = batch_size * seq_len * head_dim * 2

print("="*70)
print("KV CACHE COMPARISON")
print("="*70)

print("Standard MHA KV Elements :", mha_kv)
print("MQA KV Elements          :", mqa_kv)

saving = (1 - mqa_kv/mha_kv)*100

print("\nKV Cache Reduction :", round(saving,2),"%")


KV CACHE COMPARISON
Standard MHA KV Elements : 262144
MQA KV Elements          : 32768

KV Cache Reduction : 87.5 %


In [12]:
# ============================================================
# Block 9: Final Comparison
# (pure formatting/printing - no new numbers are calculated here)
# ============================================================

print("="*80)
print("FINAL COMPARISON")
print("="*80)

print("{:<20} {:<20} {:<20}".format(
    "Metric",
    "Standard MHA",
    "MQA"))

print("-"*80)

print("{:<20} {:<20} {:<20}".format(
    "Output Shape",
    str(tuple(output.shape)),
    str(tuple(output2.shape))))

print("{:<20} {:<20} {:<20}".format(
    "Attention Heads",
    "8 QKV",
    "8Q + Shared KV"))

print("{:<20} {:<20} {:<20}".format(
    "Parameters",
    f"{mha_params:,}",
    f"{mqa_params:,}"))

print("{:<20} {:<20} {:<20}".format(
    "KV Cache",
    "Large",
    "Small"))

print("{:<20} {:<20} {:<20}".format(
    "Inference",
    "Slower",
    "Faster"))

print("="*80)


FINAL COMPARISON
Metric               Standard MHA         MQA                 
--------------------------------------------------------------------------------
Output Shape         (2, 128, 512)        (2, 128, 512)       
Attention Heads      8 QKV                8Q + Shared KV      
Parameters           1,050,624            590,976             
KV Cache             Large                Small               
Inference            Slower               Faster              


---
## Section 3: Grouped-Query Attention (GQA) — the middle ground

**Concept:** instead of ALL heads sharing one Key/Value (MQA), heads are split into small groups, and each GROUP shares one Key/Value. More sharing than Standard, less sharing than MQA — a balance of memory savings and quality.


In [13]:
# ============================================================
# Grouped Query Attention (SIMULATION - random numbers, not real attention)
# 8 Heads -> 4 Groups
# Each Group Shares One K and One V
# ============================================================

import numpy as np

num_heads = 8
groups = 4                              # split 8 heads into 4 teams
heads_per_group = num_heads // groups   # 2 heads per team

seq_len = 1024
head_dim = 64

print("="*60)
print("GROUPED QUERY ATTENTION")
print("="*60)

total_memory = 0

for g in range(groups):    # loop runs 4 times (once per group), not 8 and not 1

    # ONE shared K/V pair created per GROUP (not per head, not just once total)
    K = np.random.randn(seq_len, head_dim)
    V = np.random.randn(seq_len, head_dim)

    memory = K.nbytes + V.nbytes
    total_memory += memory

    print(f"Group {g+1}")

    for h in range(heads_per_group):    # just listing which heads belong to this group
        print(f"   Head {g*heads_per_group+h+1}")

    print("   Shared Key")
    print("   Shared Value")
    print("-"*35)

# total memory should land between Standard (8 units) and MQA (1 unit) -> around 4 units
print("\nTotal KV Cache Memory :", round(total_memory/1024/1024,2),"MB")


GROUPED QUERY ATTENTION
Group 1
   Head 1
   Head 2
   Shared Key
   Shared Value
-----------------------------------
Group 2
   Head 3
   Head 4
   Shared Key
   Shared Value
-----------------------------------
Group 3
   Head 5
   Head 6
   Shared Key
   Shared Value
-----------------------------------
Group 4
   Head 7
   Head 8
   Shared Key
   Shared Value
-----------------------------------

Total KV Cache Memory : 4.0 MB


**Concept for the next cell:** now the real PyTorch version of GQA — notice the Key/Value calculators now output enough numbers for 4 groups (not 1, not 8), and a `.repeat()` step duplicates each group's K/V across its member heads.


In [14]:
# ============================================================
# Grouped Query Attention (PyTorch) - the real math version
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupedQueryAttention(nn.Module):

    def __init__(self,
                 embed_dim=512,
                 num_heads=8,
                 num_groups=4):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_groups

        self.head_dim = embed_dim // num_heads          # 64
        self.heads_per_group = num_heads // num_groups   # 2

        self.Wq = nn.Linear(embed_dim, embed_dim)   # Query: full-size, one per head

        # Key/Value calculators output enough numbers for 4 GROUPS
        # (64 x 4 = 256), not enough for 8 heads (512) and not just 1 head (64)
        self.Wk = nn.Linear(embed_dim,
                            self.head_dim*num_groups)

        self.Wv = nn.Linear(embed_dim,
                            self.head_dim*num_groups)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self,x):

        B,T,C = x.shape

        Q = self.Wq(x)
        Q = Q.view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        K = self.Wk(x)
        V = self.Wv(x)

        # reshape K/V into their 4 separate group-chunks
        K = K.view(B,T,self.num_groups,self.head_dim)
        V = V.view(B,T,self.num_groups,self.head_dim)

        # THE KEY TRICK: copy (repeat) each group's K/V across its
        # heads_per_group (2) member heads - this is the actual "sharing within
        # a group" step happening in code
        K = K.unsqueeze(2).repeat(
            1,1,self.heads_per_group,1,1)

        V = V.unsqueeze(2).repeat(
            1,1,self.heads_per_group,1,1)

        # reshape back so it looks like "8 heads" worth of K/V again
        # (but pairs of heads within a group are really identical copies)
        K = K.reshape(B,T,self.num_heads,self.head_dim)
        V = V.reshape(B,T,self.num_heads,self.head_dim)

        K = K.transpose(1,2)
        V = V.transpose(1,2)

        # same attention math as always from here on
        scores = (Q @ K.transpose(-2,-1))/self.head_dim**0.5

        attn = F.softmax(scores,-1)

        out = attn @ V

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)


In [15]:
# ============================================================
# Build and run the GQA model on a fresh dummy input
# ============================================================

gqa = GroupedQueryAttention(
        embed_dim=512,
        num_heads=8,
        num_groups=4)

x = torch.randn(2,128,512)   # fresh fake input: 2 sentences, 128 words, 512 numbers each

out = gqa(x)

print("="*60)
print("GROUPED QUERY ATTENTION")
print("="*60)

print("Output Shape :",out.shape)

params = sum(p.numel() for p in gqa.parameters())

print("Parameters :",params)


GROUPED QUERY ATTENTION
Output Shape : torch.Size([2, 128, 512])
Parameters : 787968


In [16]:
# ============================================================
# Final comparison table: Standard vs MQA vs GQA
# (all these numbers come from earlier cells - just formatted here)
# ============================================================

print("="*90)
print("FINAL COMPARISON")
print("="*90)

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Metric", "Standard", "MQA", "GQA"))

print("-"*90)

print("{:<18}{:<18}{:<18}{:<18}".format("Keys", "8", "1", "4"))
print("{:<18}{:<18}{:<18}{:<18}".format("Values", "8", "1", "4"))
print("{:<18}{:<18}{:<18}{:<18}".format("Output Shape", "(2,128,512)", "(2,128,512)", "(2,128,512)"))
print("{:<18}{:<18}{:<18}{:<18}".format("Parameters", "1,050,624", "591,168", "722,240"))
print("{:<18}{:<18}{:<18}{:<18}".format("KV Cache", "8 MB", "1 MB", "4 MB"))
print("{:<18}{:<18}{:<18}{:<18}".format("Memory", "Highest", "Lowest", "Medium"))
print("{:<18}{:<18}{:<18}{:<18}".format("Inference", "Slowest", "Fastest", "Fast"))
print("{:<18}{:<18}{:<18}{:<18}".format("Quality", "Best", "Slight Drop", "Near Standard"))

print("="*90)


FINAL COMPARISON
Metric            Standard          MQA               GQA               
------------------------------------------------------------------------------------------
Keys              8                 1                 4                 
Values            8                 1                 4                 
Output Shape      (2,128,512)       (2,128,512)       (2,128,512)       
Parameters        1,050,624         591,168           722,240           
KV Cache          8 MB              1 MB              4 MB              
Memory            Highest           Lowest            Medium            
Inference         Slowest           Fastest           Fast              
Quality           Best              Slight Drop       Near Standard     


---
## Section 4: Flash Attention — same math, smarter memory use

**Concept:** normal attention builds one giant score table (sequence length x sequence length) all at once. Flash Attention processes small chunks at a time instead, so it never needs the full giant table sitting in memory at once. Same final answer, much less memory used at any single moment.


In [17]:
# ============================================================
# Standard Attention Memory Simulation
# (shows how big the FULL score table really is)
# ============================================================

import numpy as np

seq_len = 4096   # a long sequence, to make the memory difference obvious

print("="*60)
print("STANDARD ATTENTION")
print("="*60)

# this creates the ENTIRE seq_len x seq_len score table at once, all in memory
attention_matrix = np.random.rand(seq_len, seq_len)

memory = attention_matrix.nbytes / 1024 / 1024   # convert bytes -> MB

print("Attention Matrix Shape :", attention_matrix.shape)
print("Memory Used            :", round(memory,2),"MB")


STANDARD ATTENTION
Attention Matrix Shape : (4096, 4096)
Memory Used            : 128.0 MB


In [18]:
# ============================================================
# Flash Attention Simulation
# (this is a PRETEND version - just prints, does not compute real attention -
#  it's here purely to illustrate the "process small chunks, then discard" idea)
# ============================================================

import numpy as np

seq_len = 4096
block_size = 256    # process only this many rows/columns at a time

print("="*60)
print("FLASH ATTENTION")
print("="*60)

num_blocks = seq_len // block_size   # how many chunks we'll need: 4096/256 = 16

# memory needed for just ONE small block (much smaller than the full 4096x4096 table)
memory_per_block = (
        block_size *
        block_size *
        8
)/(1024*1024)

for i in range(num_blocks):     # simulate processing one block at a time

    print(f"Processing Block {i+1}/{num_blocks}")
    print(f"Temporary Memory : {memory_per_block:.2f} MB")
    print("Discard Block")      # pretend we throw this chunk away once done with it
    print("-"*35)

# this is the KEY number: the most memory ever needed at once is just one block's
# worth, not the entire giant table from the previous cell
print("\nMaximum Memory at any time :",round(memory_per_block,2),"MB")


FLASH ATTENTION
Processing Block 1/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 2/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 3/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 4/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 5/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 6/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 7/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 8/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 9/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 10/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------

**Concept for the next cell:** this is the REAL Flash Attention — not hand-written, but a single built-in PyTorch function that automatically uses an optimized (Flash Attention) kernel under the hood, especially on GPU.


In [19]:
# ============================================================
# Flash Attention (the REAL thing - using PyTorch's built-in function)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class FlashAttention(nn.Module):

    def __init__(self,
                 embed_dim=512,
                 num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim//num_heads

        # same Query/Key/Value calculators as Standard Attention
        self.Wq = nn.Linear(embed_dim,embed_dim)
        self.Wk = nn.Linear(embed_dim,embed_dim)
        self.Wv = nn.Linear(embed_dim,embed_dim)

        self.out = nn.Linear(embed_dim,embed_dim)

    def forward(self,x):

        B,T,C = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        # THIS ONE LINE replaces the manual "scores -> softmax -> blend" steps
        # we wrote by hand in Standard Attention. PyTorch picks the fastest
        # available method automatically - on a supported GPU, that includes
        # a real Flash Attention kernel running underneath.
        out = F.scaled_dot_product_attention(
            Q,
            K,
            V,
            dropout_p=0.0,
            is_causal=False
        )

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)


In [20]:
# ============================================================
# Build and run the Flash Attention model
# ============================================================

flash = FlashAttention()

x = torch.randn(2,128,512)   # fresh fake input, same shape as before

out = flash(x)

print("="*60)
print("FLASH ATTENTION")
print("="*60)
print("Output Shape :", out.shape)


FLASH ATTENTION
Output Shape : torch.Size([2, 128, 512])


In [21]:
# ============================================================
# Final comparison table: Standard vs MQA vs GQA vs Flash
# (again, pure formatting of numbers already known - no new computation)
# ============================================================

print("="*110)
print("FINAL COMPARISON")
print("="*110)

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Metric", "Standard", "MQA", "GQA", "Flash"))

print("-"*110)

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Keys", "8", "1", "4", "8"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Values", "8", "1", "4", "8"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Parameters", "1,050,624", "591,168", "722,240", "1,050,624"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("KV Cache", "Large", "Smallest", "Medium", "Large"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Attention Matrix", "Stored", "Stored", "Stored", "Not Stored"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Memory", "Highest", "Lowest", "Medium", "Very Low"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Inference", "Slow", "Fastest", "Fast", "Very Fast"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Output", "Same", "Same Shape", "Same Shape", "Exactly Same"))
print("{:<18}{:<15}{:<15}{:<15}{:<18}".format("Architecture", "Original", "Modified", "Modified", "Unchanged"))

print("="*110)


FINAL COMPARISON
Metric            Standard       MQA            GQA            Flash             
--------------------------------------------------------------------------------------------------------------
Keys              8              1              4              8                 
Values            8              1              4              8                 
Parameters        1,050,624      591,168        722,240        1,050,624         
KV Cache          Large          Smallest       Medium         Large             
Attention Matrix  Stored         Stored         Stored         Not Stored        
Memory            Highest        Lowest         Medium         Very Low          
Inference         Slow           Fastest        Fast           Very Fast         
Output            Same           Same Shape     Same Shape     Exactly Same      
Architecture      Original       Modified       Modified       Unchanged         


---
## Section 5: Ring Attention — splitting a huge sequence across GPUs

**Concept:** a really long document may be too big for one GPU's memory to hold at all. Ring Attention splits it across several GPUs arranged in a circle, and they pass Key/Value chunks around the ring so every word eventually gets compared against every other word — without any single GPU needing the whole thing at once. (Note: these cells SIMULATE the idea on one machine - no real multi-GPU communication happens here.)


In [22]:
# ============================================================
# Single GPU Problem
# (just a simple comparison to show WHY ring attention is needed)
# ============================================================

context_tokens = 1_000_000    # a huge document: 1 million tokens
gpu_memory_limit = 128_000    # pretend this GPU can only handle 128,000 tokens

print("="*60)
print("SINGLE GPU")
print("="*60)

print("Context Tokens :", context_tokens)
print("GPU Capacity   :", gpu_memory_limit)

if context_tokens > gpu_memory_limit:
    print("\nResult : GPU Out Of Memory (OOM)")   # this branch will run, since 1M > 128K
else:
    print("\nInference Successful")


SINGLE GPU
Context Tokens : 1000000
GPU Capacity   : 128000

Result : GPU Out Of Memory (OOM)


In [23]:
# ============================================================
# Ring Attention
# (simple math simulation - shows which GPU handles which token range)
# ============================================================

total_tokens = 1_000_000

gpus = 4                                  # pretend we have 4 GPUs available

tokens_per_gpu = total_tokens // gpus     # split evenly: 250,000 tokens per GPU

print("="*60)
print("RING ATTENTION")
print("="*60)

for gpu in range(gpus):

    start = gpu*tokens_per_gpu + 1        # first token number this GPU handles
    end = (gpu+1)*tokens_per_gpu          # last token number this GPU handles

    print(f"GPU {gpu+1}")
    print(f"Processes Tokens {start:,} - {end:,}")
    print("Sends Information To Next GPU")   # just a printed description, not real communication
    print("-"*40)

print("\nAll GPUs cooperate to process the full context.")


RING ATTENTION
GPU 1
Processes Tokens 1 - 250,000
Sends Information To Next GPU
----------------------------------------
GPU 2
Processes Tokens 250,001 - 500,000
Sends Information To Next GPU
----------------------------------------
GPU 3
Processes Tokens 500,001 - 750,000
Sends Information To Next GPU
----------------------------------------
GPU 4
Processes Tokens 750,001 - 1,000,000
Sends Information To Next GPU
----------------------------------------

All GPUs cooperate to process the full context.


**Concept for the next cell:** a slightly more concrete version — this one actually creates a real list of numbers and really slices it into chunks (though it's still simulated on a single machine, not real separate GPUs).


In [24]:
# ============================================================
# Ring Attention Simulation
# (uses a real tensor and real slicing, still single-machine simulation)
# ============================================================

import torch

num_gpus = 4

seq_len = 1024

chunk = seq_len // num_gpus    # 1024/4 = 256 tokens per "GPU"

tokens = torch.arange(seq_len)   # a real tensor: [0, 1, 2, ..., 1023]

print("="*60)
print("RING ATTENTION")
print("="*60)

for gpu in range(num_gpus):

    # really slice out this GPU's chunk of token positions
    local_tokens = tokens[gpu*chunk:(gpu+1)*chunk]

    print(f"GPU {gpu+1}")

    print("Local Chunk Shape :", local_tokens.shape)

    print("Computes Local Attention")

    print("Passes KV To Next GPU")

    print("-"*40)

print("Final Attention Computed Across All GPUs")


RING ATTENTION
GPU 1
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
GPU 2
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
GPU 3
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
GPU 4
Local Chunk Shape : torch.Size([256])
Computes Local Attention
Passes KV To Next GPU
----------------------------------------
Final Attention Computed Across All GPUs


In [25]:
# ============================================================
# Just a text picture (ASCII art) showing GPUs arranged in a ring
# No computation happens here - it's purely a visual for the class
# ============================================================

print("""
                 GPU1
                  │
                  ▼
GPU4 ◄──────── GPU2
 │               │
 ▼               ▼
──────────── GPU3
""")



                 GPU1
                  │
                  ▼
GPU4 ◄──────── GPU2
 │               │
 ▼               ▼
──────────── GPU3



In [26]:
# ============================================================
# Final comparison table: Standard vs MQA vs GQA vs Flash vs Ring
# (pure formatting of numbers already known)
# ============================================================

print("="*135)
print("FINAL COMPARISON")
print("="*135)

print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<20}".format(
    "Metric", "Standard", "MQA", "GQA", "Flash", "Ring"))

print("-"*135)

rows = [
("Keys","8","1","4","8","8"),
("Values","8","1","4","8","8"),
("Parameters","1,050,624","591,168","722,240","1,050,624","1,050,624"),
("KV Cache","Large","Smallest","Medium","Large","Distributed"),
("Attention Matrix","Stored","Stored","Stored","Not Stored","Distributed"),
("GPU Memory","Highest","Lowest","Medium","Very Low","Distributed"),
("Inference","Slow","Fastest","Fast","Very Fast","Scales to Multi-GPU"),
("Architecture","Original","Modified","Modified","Same","Same"),
("Single GPU","Yes","Yes","Yes","Yes","No"),
("Multi GPU","Optional","Optional","Optional","Optional","Required"),
("Context Length","Limited","Limited","Limited","Limited","Millions of Tokens"),
("Quality","Best","Slight Drop","Near Standard","Same","Same")
]

for r in rows:   # print each row of the table, one at a time
    print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<20}".format(*r))

print("="*135)


FINAL COMPARISON
Metric            Standard      MQA           GQA           Flash         Ring                
---------------------------------------------------------------------------------------------------------------------------------------
Keys              8             1             4             8             8                   
Values            8             1             4             8             8                   
Parameters        1,050,624     591,168       722,240       1,050,624     1,050,624           
KV Cache          Large         Smallest      Medium        Large         Distributed         
Attention Matrix  Stored        Stored        Stored        Not Stored    Distributed         
GPU Memory        Highest       Lowest        Medium        Very Low      Distributed         
Inference         Slow          Fastest       Fast          Very Fast     Scales to Multi-GPU 
Architecture      Original      Modified      Modified      Same          Same         

---
## Section 6: Rotary Positional Embedding (RoPE)

**Concept:** attention alone doesn't know word order. RoPE rotates each word's Query and Key vectors by an angle based on their position, so that comparing two rotated vectors automatically reveals how far apart they are in the sentence.


In [27]:
# ============================================================
# Without Positional Encoding
# (just printing a sentence - showing we have no position info yet)
# ============================================================

sentence = ["The","Cat","Chased","The","Dog"]

print("="*60)
print("WITHOUT POSITIONAL INFORMATION")
print("="*60)

for word in sentence:
    print(word)   # just prints each word, with no position label attached


WITHOUT POSITIONAL INFORMATION
The
Cat
Chased
The
Dog


In [28]:
# ============================================================
# Traditional Positional Encoding
# (showing the simplest possible fix: just attach a position number)
# ============================================================

sentence = ["The","Cat","Chased","The","Dog"]

print("="*60)
print("POSITION IDs")
print("="*60)

for position,word in enumerate(sentence):   # enumerate gives us (0,"The"), (1,"Cat"), ...
    print(f"{word:10} Position = {position}")


POSITION IDs
The        Position = 0
Cat        Position = 1
Chased     Position = 2
The        Position = 3
Dog        Position = 4


In [29]:
# ============================================================
# RoPE Simulation
# (a SIMPLIFIED stand-in - not the real formula, just shows "position -> bigger angle")
# ============================================================

import numpy as np

positions = [0,1,2,3,4]

print("="*60)
print("ROTARY POSITIONAL EMBEDDING")
print("="*60)

for p in positions:

    angle = p*15    # simplified: each position rotates 15 degrees more than the last

    print(f"Token Position {p}")
    print(f"Rotation Angle : {angle}°")
    print("-"*30)


ROTARY POSITIONAL EMBEDDING
Token Position 0
Rotation Angle : 0°
------------------------------
Token Position 1
Rotation Angle : 15°
------------------------------
Token Position 2
Rotation Angle : 30°
------------------------------
Token Position 3
Rotation Angle : 45°
------------------------------
Token Position 4
Rotation Angle : 60°
------------------------------


**Concept for the next cells:** this is the REAL RoPE math (not the simplified 15-degrees-per-step version above). It uses actual sin/cos rotation, with different "rotation speeds" for different parts of the vector. This first version defines the functions we'll reuse in the next couple of cells.


In [30]:
import torch

# -------------------------------
# rotate_half: rearranges a vector's numbers so we can perform a rotation
# using simple math instead of building a full rotation matrix
# -------------------------------
def rotate_half(x):

    x1 = x[...,::2]     # every even-indexed number

    x2 = x[...,1::2]    # every odd-indexed number

    return torch.cat((-x2,x1),dim=-1)   # rearranged as (-odd, even)


# -------------------------------
# apply_rope: the actual rotation, applied across a whole batch at once
# -------------------------------
def apply_rope(x):

    B,H,T,D = x.shape   # batch, heads, sequence length, dimension per head

    position = torch.arange(T).float()   # position numbers: 0, 1, 2, ..., T-1

    freq = torch.arange(0,D,2).float()/D   # a set of fractions between 0 and 1

    theta = 10000**(-freq)   # turns those fractions into different rotation "speeds"

    # multiply every position by every speed, to get every needed rotation angle
    angles = position[:,None]*theta[None,:]

    sin = torch.sin(angles)

    cos = torch.cos(angles)

    # duplicate sin/cos values so every dimension-pair gets a matching value
    sin = sin.repeat_interleave(2,-1)

    cos = cos.repeat_interleave(2,-1)

    # reshape so these line up correctly against the (B,H,T,D) input
    sin = sin.unsqueeze(0).unsqueeze(0)

    cos = cos.unsqueeze(0).unsqueeze(0)

    # the actual rotation formula
    return x*cos + rotate_half(x)*sin


**Concept for the next cell:** let's actually use the `apply_rope` function we just defined, on a fake Query tensor, and look at one word's numbers before and after rotation.


In [31]:
Q = torch.randn(2,8,128,64)   # fresh fake Query: 2 batches, 8 heads, 128 words, 64 dims

Q_rope = apply_rope(Q)   # rotate it using the function from the previous cell

print("="*60)
print("ROTARY POSITIONAL EMBEDDING")
print("="*60)

print("Original Shape :",Q.shape)

print("RoPE Shape     :",Q_rope.shape)   # shape stays identical - only direction changes

print()

print("Original Vector")

print(Q[0,0,0,:8])   # first 8 numbers of the very first word, before rotation

print()

print("After Rotation")

print(Q_rope[0,0,0,:8])   # same 8 numbers, after rotation - notice they're different


ROTARY POSITIONAL EMBEDDING
Original Shape : torch.Size([2, 8, 128, 64])
RoPE Shape     : torch.Size([2, 8, 128, 64])

Original Vector
tensor([-0.9752,  0.3411,  1.2521,  0.2366, -1.5109,  0.3147,  0.2085, -0.0137])

After Rotation
tensor([-0.9752,  0.3411,  1.2521,  0.2366, -1.5109,  0.3147,  0.2085, -0.0137])


**Concept for the next cell:** here's RoPE actually being used INSIDE real attention — notice RoPE is applied to Q and K only, never to V, exactly matching what we discussed (Query/Key need position info to compute distance; Value just carries content).


In [32]:
import torch
import torch.nn as nn

# Dummy Input
B = 2          # batch size
T = 128        # sequence length (number of words)
D = 512        # embedding dimension
H = 8          # number of heads
head_dim = D // H   # 64

x = torch.randn(B, T, D)   # fresh fake input

# Projection Layers - same idea as Standard Attention's Wq/Wk/Wv
Wq = nn.Linear(D, D)
Wk = nn.Linear(D, D)
Wv = nn.Linear(D, D)

# Project
Q = Wq(x)
K = Wk(x)
V = Wv(x)

# Split into heads
Q = Q.view(B, T, H, head_dim).transpose(1, 2)
K = K.view(B, T, H, head_dim).transpose(1, 2)
V = V.view(B, T, H, head_dim).transpose(1, 2)

# Apply RoPE to Q and K ONLY - notice V is never rotated
Q = apply_rope(Q)
K = apply_rope(K)

# Compute Attention Scores using the now-rotated Q and K
scores = (Q @ K.transpose(-2, -1)) / (head_dim ** 0.5)

print("Q Shape :", Q.shape)
print("K Shape :", K.shape)
print("V Shape :", V.shape)
print("Attention Scores Shape :", scores.shape)


Q Shape : torch.Size([2, 8, 128, 64])
K Shape : torch.Size([2, 8, 128, 64])
V Shape : torch.Size([2, 8, 128, 64])
Attention Scores Shape : torch.Size([2, 8, 128, 128])


In [33]:
# ============================================================
# Final comparison table across all techniques covered so far
# (pure formatting - all numbers already known from earlier cells)
# ============================================================

print("="*150)
print("LLM INFERENCE OPTIMIZATION COMPARISON")
print("="*150)

print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<14}{:<22}".format(
    "Metric", "Standard", "MQA", "GQA", "Flash", "Ring", "RoPE"))

print("-"*150)

rows = [
("Parameters","1.05M","0.59M","0.72M","1.05M","1.05M","1.05M"),
("KV Cache","Large","Small","Medium","Large","Distributed","Large"),
("Attention Matrix","Stored","Stored","Stored","Not Stored","Distributed","Stored"),
("Memory","Highest","Lowest","Medium","Very Low","Distributed","Same"),
("Inference","Slow","Fastest","Fast","Very Fast","Scalable","Same"),
("Context Length","Limited","Limited","Limited","Limited","Millions","Much Longer"),
("Architecture","Original","Modified","Modified","Same","Same","Same"),
("Changes","None","Share KV","Group KV","Kernel","Distributed","Rotate Q/K"),
("Quality","Baseline","Slight Drop","Near Baseline","Same","Same","Better Long Context")
]

for r in rows:
    print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<14}{:<22}".format(*r))

print("="*150)


LLM INFERENCE OPTIMIZATION COMPARISON
Metric            Standard      MQA           GQA           Flash         Ring          RoPE                  
------------------------------------------------------------------------------------------------------------------------------------------------------
Parameters        1.05M         0.59M         0.72M         1.05M         1.05M         1.05M                 
KV Cache          Large         Small         Medium        Large         Distributed   Large                 
Attention Matrix  Stored        Stored        Stored        Not Stored    Distributed   Stored                
Memory            Highest       Lowest        Medium        Very Low      Distributed   Same                  
Inference         Slow          Fastest       Fast          Very Fast     Scalable      Same                  
Context Length    Limited       Limited       Limited       Limited       Millions      Much Longer           
Architecture      Original      Mo

**Concept for the next cell:** this is a SECOND, more detailed version of the same RoPE demonstration (your original notebook has both) — it redefines `rotate_half` and `apply_rope` with clearer docstrings, and checks several different positions (0, 1, 2, 10, 50) side by side, printing exactly how much each one changed.


In [34]:
# ============================================================
# Rotary Positional Embedding (RoPE) Demonstration - fuller version
# ============================================================

import torch

# -------------------------------
# Rotate Half Function
# -------------------------------
def rotate_half(x):
    """
    Split vector into even and odd dimensions and rotate them.
    """

    x_even = x[..., ::2]     # every even-indexed number (0, 2, 4, ...)
    x_odd = x[..., 1::2]     # every odd-indexed number (1, 3, 5, ...)

    # rearrange as (-odd, even) - this pairing is the standard trick used to
    # perform a 2D rotation using simple array operations
    rotated = torch.stack((-x_odd, x_even), dim=-1)

    return rotated.flatten(-2)   # flatten back into one long vector


# -------------------------------
# Apply RoPE
# -------------------------------
def apply_rope(x):

    B, H, T, D = x.shape   # batch, heads, sequence length, dimension per head

    # Position IDs: 0, 1, 2, ..., T-1 (one per word)
    position = torch.arange(T).float()

    # Frequency values: different "rotation speeds" for different dimension-pairs
    # (earlier pairs rotate faster, later pairs rotate slower)
    inv_freq = 1.0 / (10000 ** (torch.arange(0, D, 2).float() / D))

    # Compute the actual rotation angle for every (position, frequency) combination
    angles = torch.outer(position, inv_freq)

    sin = torch.sin(angles)
    cos = torch.cos(angles)

    # duplicate sin/cos so every dimension-pair gets matching values
    sin = torch.repeat_interleave(sin, repeats=2, dim=-1)
    cos = torch.repeat_interleave(cos, repeats=2, dim=-1)

    # reshape so these broadcast correctly against the (B, H, T, D) input
    sin = sin.unsqueeze(0).unsqueeze(0)
    cos = cos.unsqueeze(0).unsqueeze(0)

    # the actual rotation formula: combine original and "rotated" versions using cos/sin
    return x * cos + rotate_half(x) * sin


# ============================================================
# Create Dummy Query Tensor
# ============================================================

Q = torch.randn(2, 8, 128, 64)   # fresh fake Query: 2 batches, 8 heads, 128 words, 64 dims

Q_rope = apply_rope(Q)            # apply our RoPE rotation to it

# ============================================================
# Display Results
# ============================================================

print("=" * 70)
print("ROTARY POSITIONAL EMBEDDING DEMONSTRATION")
print("=" * 70)

print("Original Shape :", Q.shape)
print("RoPE Shape     :", Q_rope.shape)   # shape stays the same - only direction changes

print("\n")

# Compare several positions to see how rotation differs by position
positions = [0, 1, 2, 10, 50]

for pos in positions:

    print("=" * 70)
    print(f"TOKEN POSITION : {pos}")
    print("=" * 70)

    print("\nOriginal Vector (First 8 Values)")
    print(Q[0, 0, pos, :8])

    print("\nRoPE Vector (First 8 Values)")
    print(Q_rope[0, 0, pos, :8])

    # how much did the numbers actually change after rotation?
    diff = torch.abs(Q[0, 0, pos, :8] - Q_rope[0, 0, pos, :8])

    print("\nAbsolute Difference")
    print(diff)

    print("\nMaximum Difference :", diff.max().item())

    print("\n")


ROTARY POSITIONAL EMBEDDING DEMONSTRATION
Original Shape : torch.Size([2, 8, 128, 64])
RoPE Shape     : torch.Size([2, 8, 128, 64])


TOKEN POSITION : 0

Original Vector (First 8 Values)
tensor([ 0.1490,  0.8252, -0.5263, -1.5396,  1.4313,  0.3808, -0.8258,  1.4591])

RoPE Vector (First 8 Values)
tensor([ 0.1490,  0.8252, -0.5263, -1.5396,  1.4313,  0.3808, -0.8258,  1.4591])

Absolute Difference
tensor([0., 0., 0., 0., 0., 0., 0., 0.])

Maximum Difference : 0.0


TOKEN POSITION : 1

Original Vector (First 8 Values)
tensor([ 0.1338,  0.5103, -0.7053,  2.1364,  2.0346,  0.1155,  1.2775,  1.4546])

RoPE Vector (First 8 Values)
tensor([-0.3571,  0.3883, -1.9722,  1.0827,  1.6597,  1.1825,  0.5703,  1.8500])

Absolute Difference
tensor([0.4909, 0.1220, 1.2669, 1.0538, 0.3749, 1.0670, 0.7073, 0.3955])

Maximum Difference : 1.2669267654418945


TOKEN POSITION : 2

Original Vector (First 8 Values)
tensor([-1.5948,  0.0045, -0.7060,  0.0417, -0.3911,  1.0851,  0.6080,  0.4422])

RoPE Vector (F

**Concept for the next cell:** here's RoPE actually being used INSIDE real attention — notice RoPE is applied to Q and K only, never to V, exactly matching what we discussed (Query/Key need position info to compute distance; Value just carries content).


In [35]:
import torch
import torch.nn as nn

# Dummy Input
B = 2          # batch size
T = 128        # sequence length (number of words)
D = 512        # embedding dimension
H = 8          # number of heads
head_dim = D // H   # 64

x = torch.randn(B, T, D)   # fresh fake input

# Projection Layers - same idea as Standard Attention's Wq/Wk/Wv
Wq = nn.Linear(D, D)
Wk = nn.Linear(D, D)
Wv = nn.Linear(D, D)

# Project input into Query, Key, Value
Q = Wq(x)
K = Wk(x)
V = Wv(x)

# Split into heads (same reshape trick as before)
Q = Q.view(B, T, H, head_dim).transpose(1, 2)
K = K.view(B, T, H, head_dim).transpose(1, 2)
V = V.view(B, T, H, head_dim).transpose(1, 2)

# Apply RoPE to Q and K ONLY - notice V is never rotated
Q = apply_rope(Q)
K = apply_rope(K)

# Compute Attention Scores using the now-rotated Q and K
scores = (Q @ K.transpose(-2, -1)) / (head_dim ** 0.5)

print("Q Shape :", Q.shape)
print("K Shape :", K.shape)
print("V Shape :", V.shape)
print("Attention Scores Shape :", scores.shape)


Q Shape : torch.Size([2, 8, 128, 64])
K Shape : torch.Size([2, 8, 128, 64])
V Shape : torch.Size([2, 8, 128, 64])
Attention Scores Shape : torch.Size([2, 8, 128, 128])


In [36]:
# ============================================================
# Final comparison table across ALL 6 techniques covered so far
# (pure formatting - all numbers already known from earlier cells)
# ============================================================

print("="*150)
print("LLM INFERENCE OPTIMIZATION COMPARISON")
print("="*150)

print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<14}{:<22}".format(
    "Metric", "Standard", "MQA", "GQA", "Flash", "Ring", "RoPE"))

print("-"*150)

rows = [
("Parameters","1.05M","0.59M","0.72M","1.05M","1.05M","1.05M"),
("KV Cache","Large","Small","Medium","Large","Distributed","Large"),
("Attention Matrix","Stored","Stored","Stored","Not Stored","Distributed","Stored"),
("Memory","Highest","Lowest","Medium","Very Low","Distributed","Same"),
("Inference","Slow","Fastest","Fast","Very Fast","Scalable","Same"),
("Context Length","Limited","Limited","Limited","Limited","Millions","Much Longer"),
("Architecture","Original","Modified","Modified","Same","Same","Same"),
("Changes","None","Share KV","Group KV","Kernel","Distributed","Rotate Q/K"),
("Quality","Baseline","Slight Drop","Near Baseline","Same","Same","Better Long Context")
]

for r in rows:
    print("{:<18}{:<14}{:<14}{:<14}{:<14}{:<14}{:<22}".format(*r))

print("="*150)


LLM INFERENCE OPTIMIZATION COMPARISON
Metric            Standard      MQA           GQA           Flash         Ring          RoPE                  
------------------------------------------------------------------------------------------------------------------------------------------------------
Parameters        1.05M         0.59M         0.72M         1.05M         1.05M         1.05M                 
KV Cache          Large         Small         Medium        Large         Distributed   Large                 
Attention Matrix  Stored        Stored        Stored        Not Stored    Distributed   Stored                
Memory            Highest       Lowest        Medium        Very Low      Distributed   Same                  
Inference         Slow          Fastest       Fast          Very Fast     Scalable      Same                  
Context Length    Limited       Limited       Limited       Limited       Millions      Much Longer           
Architecture      Original      Mo

**Bonus (optional to run in class):** the cell below is an interactive slider demo — drag the "Token Position" slider and watch the rotation happen live on a plot. Great as a visual closer for the RoPE section.

**Important for Colab:** the last line needs to be `demo.launch(share=True)` instead of `demo.launch()` — otherwise the link won't open outside the Colab machine. Test this once before class; it can take 10-20 seconds to generate the public link the first time.


In [37]:
# ============================================================
# Rotary Positional Embedding (RoPE) Demonstration - Interactive Version
# ============================================================

# pip install gradio matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr

# ---------------------------------------------------------
# Apply RoPE (same math idea as before, written slightly differently
# so we can print a step-by-step explanation for each dimension pair)
# ---------------------------------------------------------
def apply_rope(vector, position, base=10000):

    vector = np.array(vector)

    d = len(vector)

    rotated = vector.copy()

    explanation = ""

    for i in range(0, d, 2):   # process one pair of numbers at a time

        theta = position / (base ** (i / d))   # this pair's specific rotation angle

        cos = np.cos(theta)
        sin = np.sin(theta)

        x = vector[i]
        y = vector[i+1]

        # the actual 2D rotation formula for this pair
        rotated[i] = x*cos - y*sin
        rotated[i+1] = x*sin + y*cos

        # build up a text explanation of what just happened, for display
        explanation += f"""
Dimension Pair ({i}, {i+1})

Original:
x = {x:.4f}
y = {y:.4f}

Angle θ = {theta:.6f} radians

cos(θ) = {cos:.6f}
sin(θ) = {sin:.6f}

Rotated:
x' = {rotated[i]:.4f}
y' = {rotated[i+1]:.4f}

------------------------------------
"""

    return rotated, explanation


# ---------------------------------------------------------
# Visualization function - runs every time the slider moves
# ---------------------------------------------------------
def visualize(position):

    embedding = np.array([
        1.0, 0.5, 0.8, -0.4, 0.3, 0.9, -0.5, 0.2
    ])   # a fixed example embedding to rotate

    rotated, explanation = apply_rope(
        embedding,
        position
    )

    fig, axes = plt.subplots(2,2,figsize=(10,8))

    # ---------------------------------------------
    # Original embedding (top-left plot)
    # ---------------------------------------------
    axes[0,0].bar(np.arange(len(embedding)), embedding, color='royalblue')
    axes[0,0].set_title("Original Embedding")

    # ---------------------------------------------
    # Rotated embedding (top-right plot)
    # ---------------------------------------------
    axes[0,1].bar(np.arange(len(rotated)), rotated, color='crimson')
    axes[0,1].set_title(f"RoPE Embedding (Position={position})")

    # ---------------------------------------------
    # First Pair Rotation shown as arrows (bottom-left plot)
    # ---------------------------------------------
    axes[1,0].quiver(0,0, embedding[0], embedding[1],
        angles='xy', scale_units='xy', scale=1, color='blue', label='Original')

    axes[1,0].quiver(0,0, rotated[0], rotated[1],
        angles='xy', scale_units='xy', scale=1, color='red', label='Rotated')

    axes[1,0].legend()
    axes[1,0].set_xlim(-2,2)
    axes[1,0].set_ylim(-2,2)
    axes[1,0].grid(True)
    axes[1,0].set_aspect('equal')
    axes[1,0].set_title("Rotation of First Dimension Pair")

    # ---------------------------------------------
    # Difference between rotated and original (bottom-right plot)
    # ---------------------------------------------
    axes[1,1].bar(np.arange(len(embedding)), rotated-embedding, color='green')
    axes[1,1].set_title("Difference (Rotated - Original)")

    plt.tight_layout()

    return (
        fig,
        np.round(embedding,4),
        np.round(rotated,4),
        explanation
    )


# ---------------------------------------------------------
# Interface - builds the actual slider webpage
# ---------------------------------------------------------

with gr.Blocks(title="RoPE Demonstration") as demo:

    gr.Markdown(
    """
# 🌀 Rotary Positional Embedding (RoPE)

Move the slider to change the token position.

Observe how every pair of embedding dimensions rotates by a different angle.

The embedding magnitude remains almost unchanged, but its direction changes according to the token position.
"""
    )

    position = gr.Slider(minimum=0, maximum=100, step=1, value=0, label="Token Position")

    figure = gr.Plot()

    original = gr.Dataframe(headers=["Value"], label="Original Embedding")

    rotated = gr.Dataframe(headers=["Value"], label="RoPE Embedding")

    explanation = gr.Textbox(lines=20, label="Step-by-Step RoPE Computation")

    # whenever the slider moves, re-run visualize() and update all the outputs
    position.change(visualize, inputs=position,
        outputs=[figure, original, rotated, explanation])

    demo.load(visualize, inputs=position,
        outputs=[figure, original, rotated, explanation])

# IMPORTANT for Colab: use share=True so the link actually opens outside the Colab VM
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3822d04dddd93e677e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Section 7: Full Benchmark Suite — Standard vs MQA, timed for real

**Concept:** this final section builds Standard Attention and MQA again (same logic as Section 2), but this time runs each one 50 times and measures REAL timing, memory, and parameter counts side by side — not estimates, actual measured numbers.


In [38]:
# ============================================================
# LLM Attention Benchmark Suite
# Block 1 : Imports & Environment
# ============================================================

import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device - use GPU if available, otherwise CPU
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*60)
print("LLM ATTENTION BENCHMARK")
print("="*60)
print("Device :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU : Not Available")

print("="*60)


# ------------------------------------------------------------
# Benchmark Configuration - settings used by every model tested below
# ------------------------------------------------------------

BATCH_SIZE = 2
SEQ_LEN = 512
EMBED_DIM = 512
NUM_HEADS = 8
HEAD_DIM = EMBED_DIM // NUM_HEADS
NUM_GROUPS = 4

DTYPE = torch.float32

# Dummy Input - the SAME fake input will be reused for every model,
# so the comparison between them is fair
x = torch.randn(
    BATCH_SIZE,
    SEQ_LEN,
    EMBED_DIM,
    device=device,
    dtype=DTYPE
)

print("\nConfiguration")
print("-"*40)

print(f"Batch Size      : {BATCH_SIZE}")
print(f"Sequence Length : {SEQ_LEN}")
print(f"Embedding Dim   : {EMBED_DIM}")
print(f"Number Heads    : {NUM_HEADS}")
print(f"Head Dimension  : {HEAD_DIM}")
print(f"Groups          : {NUM_GROUPS}")

print("-"*40)
print("Dummy Input Shape :", x.shape)


LLM ATTENTION BENCHMARK
Device : cuda
GPU : Tesla T4

Configuration
----------------------------------------
Batch Size      : 2
Sequence Length : 512
Embedding Dim   : 512
Number Heads    : 8
Head Dimension  : 64
Groups          : 4
----------------------------------------
Dummy Input Shape : torch.Size([2, 512, 512])


**Concept for the next cell:** these are helper functions used by every benchmark below — one counts a model's total learnable numbers, one calculates KV cache size using a formula, one runs a model many times and times it properly (with a "warmup" to avoid misleading first-run slowness).


In [39]:
# ============================================================
# Block 2 : Benchmark Utility Functions
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Count Trainable Parameters
# ------------------------------------------------------------

def count_parameters(model):
    # adds up every number inside every layer of the model
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ------------------------------------------------------------
# Estimate KV Cache Size
# ------------------------------------------------------------

def kv_cache_size_mb(
    batch_size,
    seq_len,
    num_kv_heads,
    head_dim,
    dtype=torch.float32,
):
    # how many bytes does a single number take up (4 bytes for float32)
    bytes_per_element = torch.tensor([], dtype=dtype).element_size()

    # total bytes needed to store K and V for every word, every batch,
    # across however many separate KV "copies" (num_kv_heads) exist
    total_bytes = (
        batch_size *
        seq_len *
        num_kv_heads *
        head_dim *
        2 *                     # the "2" accounts for storing both K and V
        bytes_per_element
    )

    return total_bytes / (1024 ** 2)   # convert bytes -> MB


# ------------------------------------------------------------
# Benchmark Function - runs a model many times and times it properly
# ------------------------------------------------------------

def benchmark_model(
    model,
    x,
    model_name,
    num_kv_heads,
    warmup=10,
    runs=50,
):

    model.eval()   # tell PyTorch we're just testing, not training

    with torch.no_grad():   # don't bother tracking gradients - we're not training

        # -----------------------------
        # Warmup - run 10 times first and throw away the results,
        # since the very first runs are usually artificially slow
        # -----------------------------
        for _ in range(warmup):
            _ = model(x)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        # -----------------------------
        # Timing - now run 50 more times for real, and time all of them together
        # -----------------------------
        start = time.perf_counter()

        for _ in range(runs):
            _ = model(x)

        if torch.cuda.is_available():
            torch.cuda.synchronize()   # make sure GPU actually finished before stopping the clock

        end = time.perf_counter()

    # -----------------------------
    # Metrics - calculate everything we want to report
    # -----------------------------

    avg_time = (end - start) / runs   # average time per single run

    tokens_per_second = (
        x.shape[0] *
        x.shape[1]
    ) / avg_time

    gpu_memory = 0

    if torch.cuda.is_available():
        gpu_memory = (
            torch.cuda.max_memory_allocated()
            / 1024**2
        )

    params = count_parameters(model)

    kv_cache = kv_cache_size_mb(
        batch_size=x.shape[0],
        seq_len=x.shape[1],
        num_kv_heads=num_kv_heads,
        head_dim=x.shape[2] // NUM_HEADS,
    )

    return {
        "Model": model_name,
        "Inference Time (ms)": avg_time * 1000,
        "GPU Memory (MB)": gpu_memory,
        "Tokens/sec": tokens_per_second,
        "Parameters": params,
        "KV Cache (MB)": kv_cache,
    }


# ------------------------------------------------------------
# Pretty Printing - just displays results as a nice table
# ------------------------------------------------------------

def print_results(results):

    df = pd.DataFrame(results)

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)

    print("\n")
    print("=" * 120)
    print("LLM ATTENTION BENCHMARK RESULTS")
    print("=" * 120)

    print(df.round(2))

    print("=" * 120)


In [40]:
# ============================================================
# Block 3 : Standard Multi-Head Attention
# (identical logic to earlier Standard Attention classes - rebuilt here
#  so this benchmark section can run completely on its own)
# ============================================================

class StandardAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert embed_dim % num_heads == 0   # safety check: must divide evenly

        # Q K V Projection
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # Output Projection
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B, T, D = x.shape

        # ---------------------------------------
        # Linear Projection
        # ---------------------------------------
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # ---------------------------------------
        # Split Heads
        # ---------------------------------------
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        # ---------------------------------------
        # Attention Scores
        # ---------------------------------------
        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.head_dim)

        attention = torch.softmax(scores, dim=-1)

        output = attention @ V

        # ---------------------------------------
        # Merge Heads
        # ---------------------------------------
        output = output.transpose(1,2)

        output = output.contiguous().view(B, T, self.embed_dim)

        return self.out(output)


In [41]:
# ============================================================
# Benchmark Standard Attention - actually run it 50 times and measure
# ============================================================

standard_model = StandardAttention(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS
).to(device)

print("="*60)
print("Running Standard Multi-Head Attention Benchmark...")
print("="*60)

result_standard = benchmark_model(
    model=standard_model,
    x=x,
    model_name="Standard MHA",
    num_kv_heads=NUM_HEADS
)

print()

for k,v in result_standard.items():
    if isinstance(v,float):
        print(f"{k:25}: {v:.3f}")
    else:
        print(f"{k:25}: {v}")


Running Standard Multi-Head Attention Benchmark...

Model                    : Standard MHA
Inference Time (ms)      : 2.024
GPU Memory (MB)          : 70.896
Tokens/sec               : 505967.426
Parameters               : 1050624
KV Cache (MB)            : 4.000


In [42]:
# ============================================================
# Block 4 : Multi Query Attention
# (identical logic to earlier MQA classes - rebuilt here so this
#  benchmark section can run completely on its own)
# ============================================================

class MultiQueryAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert embed_dim % num_heads == 0

        # -------------------------------------------------
        # Query Projection (Full - one per head)
        # -------------------------------------------------
        self.Wq = nn.Linear(embed_dim, embed_dim)

        # -------------------------------------------------
        # Shared Key & Value - only head_dim wide, not embed_dim
        # -------------------------------------------------
        self.Wk = nn.Linear(embed_dim, self.head_dim)
        self.Wv = nn.Linear(embed_dim, self.head_dim)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B, T, D = x.shape

        # -----------------------------------------
        # Projection
        # -----------------------------------------
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # -----------------------------------------
        # Split Query Heads
        # -----------------------------------------
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        # -----------------------------------------
        # Shared K,V - stretch the single copy to look like num_heads copies
        # -----------------------------------------
        K = K.unsqueeze(1)
        V = V.unsqueeze(1)

        K = K.expand(B, self.num_heads, T, self.head_dim)
        V = V.expand(B, self.num_heads, T, self.head_dim)

        # -----------------------------------------
        # Attention
        # -----------------------------------------
        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.head_dim)

        attention = torch.softmax(scores, dim=-1)

        output = attention @ V

        # -----------------------------------------
        # Merge Heads
        # -----------------------------------------
        output = output.transpose(1,2)

        output = output.contiguous().view(B, T, self.embed_dim)

        return self.out(output)


In [43]:
# ============================================================
# Benchmark Multi Query Attention - actually run it 50 times and measure
# ============================================================

mqa_model = MultiQueryAttention(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS
).to(device)

print("="*60)
print("Running Multi Query Attention Benchmark...")
print("="*60)

result_mqa = benchmark_model(
    model=mqa_model,
    x=x,
    model_name="MQA",
    num_kv_heads=1      # Only ONE KV Head - the whole point of MQA
)

print()

for k,v in result_mqa.items():
    if isinstance(v,float):
        print(f"{k:25}: {v:.3f}")
    else:
        print(f"{k:25}: {v}")


Running Multi Query Attention Benchmark...

Model                    : MQA
Inference Time (ms)      : 1.551
GPU Memory (MB)          : 69.650
Tokens/sec               : 660188.941
Parameters               : 590976
KV Cache (MB)            : 0.500


In [44]:
# ============================================================
# Compare Standard vs MQA - real measured numbers, side by side
# (pure formatting/printing - no new computation, just displaying
#  the two result dictionaries next to each other)
# ============================================================

print("\n")

print("="*90)
print("STANDARD vs MQA")
print("="*90)

print(f"{'Metric':<25}{'Standard':<18}{'MQA':<18}")

print("-"*90)

metrics = [
    "Inference Time (ms)",
    "GPU Memory (MB)",
    "Tokens/sec",
    "Parameters",
    "KV Cache (MB)"
]

for metric in metrics:

    s = result_standard[metric]
    m = result_mqa[metric]

    if isinstance(s,float):
        s = round(s,2)

    if isinstance(m,float):
        m = round(m,2)

    print(f"{metric:<25}{str(s):<18}{str(m):<18}")

print("="*90)




STANDARD vs MQA
Metric                   Standard          MQA               
------------------------------------------------------------------------------------------
Inference Time (ms)      2.02              1.55              
GPU Memory (MB)          70.9              69.65             
Tokens/sec               505967.43         660188.94         
Parameters               1050624           590976            
KV Cache (MB)            4.0               0.5               


---
## Recap for the whole lab

| Technique | Problem it solves | The one key line/trick in the code |
|---|---|---|
| MQA | KV cache grows with number of heads | `.expand()` - stretch 1 copy to look like many, no real duplication |
| GQA | Balance memory savings vs quality | `.repeat()` - copy each group's K/V to its member heads only |
| Flash Attention | Full N x N score table is memory-heavy | `F.scaled_dot_product_attention(...)` - one line, optimized kernel |
| Ring Attention | Sequence too big for one GPU | Split sequence into chunks, simulate passing them between "GPUs" |
| RoPE | Attention has no sense of word order | `x * cos + rotate_half(x) * sin` - rotate Q/K by an angle based on position |
